# SnowIn core workflow

This data-free tutorial shows SnowIn's xarray-native science: phase using the SnowIn convention, an explicit wavelength, contributor-based phase referencing, named support, and a chronological dSWE path.

Phase is `phi_reference - phi_secondary`; pairwise dSWE is `SWE_secondary - SWE_reference`. Phase and incidence are radians, and wavelength is metres. The example uses local incidence and leaves missing phase/support missing.

In [ ]:
import math

import numpy as np
import xarray as xr

from snowin import (
    accumulate_dswe,
    build_support_dataset,
    compose_support_mask,
    compute_leinss_dswe,
    reference_phase,
    summarize_support,
)

## 1. Prepare an xarray pair

The adapter is omitted here: callers can supply phase and incidence as ordinary xarray objects, with their scientific conventions recorded in attributes.

In [ ]:
coords = {
    "y": [4200.0, 4190.0],
    "x": [500000.0, 500010.0, 500020.0],
    "spatial_ref": xr.DataArray(
        0, attrs={"epsg_code": 32613, "spatial_ref": "EPSG:32613"}
    ),
}
phase = xr.DataArray(
    [[0.55, 0.62, 0.70], [0.48, np.nan, 0.66]],
    dims=("y", "x"),
    coords=coords,
    name="phase",
    attrs={
        "units": "rad",
        "phase_difference_definition": "reference_minus_secondary",
        "grid_mapping": "spatial_ref",
    },
)
incidence = xr.DataArray(
    np.full(phase.shape, math.radians(35.0)),
    dims=phase.dims,
    coords=coords,
    name="incidence_angle",
    attrs={
        "units": "rad",
        "incidence_angle_reference": "local",
        "grid_mapping": "spatial_ref",
    },
)
pair = xr.Dataset(
    {"phase": phase, "incidence_angle": incidence},
    attrs={
        "phase_difference_definition": "reference_minus_secondary",
        "temporal_edge": "reference_to_secondary",
        "reference_time": "2025-01-01T00:00:00Z",
        "secondary_time": "2025-01-13T00:00:00Z",
    },
)
pair

## 2. Compute pairwise dSWE

Choose a named physical method and supply the wavelength explicitly. Its sign follows the directed pair: secondary SWE minus reference SWE.

In [ ]:
wavelength_m = 0.24
pairwise_dswe = compute_leinss_dswe(
    pair.phase,
    pair.incidence_angle,
    wavelength_m=wavelength_m,
)
pairwise_dswe

## 3. Apply an auditable phase reference

The caller supplies observed phase, expected phase, and weights for each contributor. SnowIn estimates the offset and records its contributors; station selection and expected-phase construction remain caller-owned.

In [ ]:
station = ["site-a", "site-b", "site-c"]
observed_phase = xr.DataArray(
    [0.42, 0.31, 0.50], dims="station", attrs={"units": "rad", "phase_difference_definition": "reference_minus_secondary"}
)
expected_phase = xr.DataArray(
    [0.12, 0.11, 0.20], dims="station", attrs={"units": "rad", "phase_difference_definition": "reference_minus_secondary"}
)
weights = xr.DataArray([1.0, 2.0, 1.0], dims="station", attrs={"units": "1"})
reference = reference_phase(
    pair,
    observed_phase,
    expected_phase,
    weights,
    contributor_id=xr.DataArray(station, dims="station"),
)
print("reference status:", reference.attrs["phase_reference_status"])
print("reference offset (rad):", reference.attrs["phase_reference_offset_rad"])
reference[["phase_referenced", "reference_contributor_id", "reference_eligible"]]

## 4. Keep support named and explicit

Product validity and geometry validity are separate evidence. Compose only the support required for this pairwise operation; missing phase remains unsupported.

In [ ]:
support = build_support_dataset(
    product_valid=pair.phase.notnull().rename("product_valid"),
    geometry_valid=pair.incidence_angle.notnull().rename("geometry_valid"),
)
pairwise_supported = compose_support_mask(
    support, ["product_valid", "geometry_valid"], name="pairwise_supported"
)
print("support summary:")
print(summarize_support(support))
print("explicitly composed pairwise support:")
pairwise_supported

## 5. Accumulate an explicit chronological path

The second synthetic edge starts exactly when the first ends. Accumulation preserves unsupported samples instead of treating them as zero.

In [ ]:
edge_1 = xr.Dataset(
    {
        "dswe": pairwise_dswe.where(pairwise_supported),
        "pairwise_supported": pairwise_supported,
    },
    attrs={
        "temporal_edge": "reference_to_secondary",
        "reference_time": "2025-01-01T00:00:00Z",
        "secondary_time": "2025-01-13T00:00:00Z",
    },
)
edge_1.dswe.attrs.update({"units": "m", "quantity": "pairwise_dSWE"})
edge_2_dswe = (edge_1.dswe * 0.5).assign_attrs(edge_1.dswe.attrs)
edge_2 = xr.Dataset(
    {"dswe": edge_2_dswe, "pairwise_supported": pairwise_supported},
    attrs={
        "temporal_edge": "reference_to_secondary",
        "reference_time": "2025-01-13T00:00:00Z",
        "secondary_time": "2025-01-25T00:00:00Z",
    },
)
path = accumulate_dswe([edge_1, edge_2])
path